In [5]:
import sys
sys.path.append(r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma\notebooks')


  Obtaining dependency information for numpy<2 from https://files.pythonhosted.org/packages/3f/6b/5610004206cf7f8e7ad91c5a85a8c71b2f2f8051a0c0c4d5916b76d6cbb2/numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-core 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.8.0 which is incompatible.
autogluon-features 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.8.0 which is incompatible.
autogluon-tabular 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.8.0 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


In [9]:
import pickle
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
from src.features import (
    calculate_three_layer_features_v2,
    calculate_strike_features,
    calculate_r1_features,
    calculate_career_stats,
    parse_reach,
    parse_height,
    parse_age,
    parse_fight_duration,
    time_decay_weights,
    kish_effective_n,
    bayesian_smooth,
    normalize_weight_class
)
#from src.predict import predict_fight
DATA_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma\notebooks\02_features\data'
DB_PATH   = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'

model                   = pickle.load(open(f'{DATA_PATH}/xgb_best_model.pkl', 'rb'))
feature_cols            = pickle.load(open(f'{DATA_PATH}/feature_cols.pkl', 'rb'))
fighter_fights_dict     = pickle.load(open(f'{DATA_PATH}/fighter_fights_dict.pkl', 'rb'))
opponents_dict          = pickle.load(open(f'{DATA_PATH}/opponents_dict.pkl', 'rb'))
fighter_adjperf_history = pickle.load(open(f'{DATA_PATH}/fighter_adjperf_history.pkl', 'rb'))
all_fight_stats         = pickle.load(open(f'{DATA_PATH}/all_fight_stats.pkl', 'rb'))
conn                    = sqlite3.connect(DB_PATH)

print("✔ All data loaded")
fighter_fights_dict     = pickle.load(open(f'{DATA_PATH}/fighter_fights_dict.pkl', 'rb'))
opponents_dict          = pickle.load(open(f'{DATA_PATH}/opponents_dict.pkl', 'rb'))
fighter_adjperf_history = pickle.load(open(f'{DATA_PATH}/fighter_adjperf_history.pkl', 'rb'))

# DB connection
conn = sqlite3.connect(DB_PATH)

print(f"✔ Model loaded")
print(f"✔ Features: {len(feature_cols)}")
print(f"✔ Fighters cached: {len(fighter_fights_dict)}")

✔ All data loaded
✔ Model loaded
✔ Features: 118
✔ Fighters cached: 3760


In [7]:
# Test that functions are importable and callable
test_result = calculate_three_layer_features_v2(
    fighter_id=list(fighter_fights_dict.keys())[0],
    opponent_id=list(fighter_fights_dict.keys())[1],
    as_of_date='2024-01-01',
    fighter_fights_dict=fighter_fights_dict,
    opponents_dict=opponents_dict,
    fighter_adjperf_history=fighter_adjperf_history,
    all_fight_stats=all_fight_stats
)

print(test_result)

NameError: name 'all_fight_stats' is not defined

In [ ]:
def predict_fight(f1_name, f2_name, conn, as_of_date=None):
    if as_of_date is None:
        as_of_date = datetime.now().strftime("%Y-%m-%d")

    fighters_lookup = pd.read_sql("SELECT fighter_id, name FROM fighters_v2", conn)

    def find_fighter(name, all_fighters):
        exact = all_fighters[all_fighters['name'].str.lower() == name.lower()]
        if len(exact) == 1:
            return exact.iloc[0]
        full = all_fighters[all_fighters['name'].str.contains(name, case=False, na=False)]
        if len(full) == 1:
            return full.iloc[0]
        if len(full) > 1:
            print(f"  WARNING: Multiple matches for '{name}':")
            print(f"  {full[['fighter_id','name']].to_string(index=False)}")
            return full.iloc[0]
        last = name.split()[-1]
        partial = all_fighters[all_fighters['name'].str.contains(last, case=False, na=False)]
        if len(partial) > 0:
            print(f"  WARNING: No exact match for '{name}', best guess:")
            print(f"  {partial[['fighter_id','name']].head(5).to_string(index=False)}")
            return partial.iloc[0]
        return None

    f1_match = find_fighter(f1_name, fighters_lookup)
    f2_match = find_fighter(f2_name, fighters_lookup)
    if f1_match is None or f2_match is None:
        print(f"ERROR: Could not find fighter(s). F1={f1_name}, F2={f2_name}")
        return None

    f1_id, f1_db_name = f1_match['fighter_id'], f1_match['name']
    f2_id, f2_db_name = f2_match['fighter_id'], f2_match['name']
    print(f"  F1: {f1_db_name} ({f1_id})")
    print(f"  F2: {f2_db_name} ({f2_id})")

    f1_feats = calculate_three_layer_features_v2(f1_id, f2_id, as_of_date)
    f2_feats = calculate_three_layer_features_v2(f2_id, f1_id, as_of_date)
    if f1_feats is None or f2_feats is None:
        print("ERROR: Could not compute base features. Check fight history.")
        return None

    f1_feats.update(calculate_strike_features(f1_id, f2_id, as_of_date))
    f2_feats.update(calculate_strike_features(f2_id, f1_id, as_of_date))
    f1_feats.update(calculate_r1_features(f1_id, f2_id, as_of_date))
    f2_feats.update(calculate_r1_features(f2_id, f1_id, as_of_date))
    f1_feats.update(calculate_career_stats(f1_id, f2_id, 'PRED', as_of_date))
    f2_feats.update(calculate_career_stats(f2_id, f1_id, 'PRED', as_of_date))

    row = {}
    for k, v in f1_feats.items(): row[f'f1_{k}'] = v
    for k, v in f2_feats.items(): row[f'f2_{k}'] = v
    for k in f1_feats.keys():     row[f'diff_{k}'] = row[f'f1_{k}'] - row[f'f2_{k}']

    fighter_info = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob FROM fighters_v2
        WHERE fighter_id IN ('{f1_id}', '{f2_id}')
    """, conn)

    def get_phys(fid):
        info = fighter_info[fighter_info['fighter_id'] == fid]
        if len(info) == 0:
            return {'reach': 72, 'height': 70, 'age': 30}
        info = info.iloc[0]
        return {
            'reach':  parse_reach(info['reach'])   or 72,
            'height': parse_height(info['height']) or 70,
            'age':    parse_age(info['dob'], as_of_date) or 30,
        }

    f1_p, f2_p = get_phys(f1_id), get_phys(f2_id)
    row['age_diff']     = f1_p['age']    - f2_p['age']
    row['age_ratio']    = f1_p['age']    / f2_p['age']
    row['reach_diff']   = f1_p['reach']  - f2_p['reach']
    row['reach_ratio']  = f1_p['reach']  / f2_p['reach']
    row['height_diff']  = f1_p['height'] - f2_p['height']
    row['height_ratio'] = f1_p['height'] / f2_p['height']

    def get_ufc_age(fid):
        if fid in fighter_fights_dict:
            first = fighter_fights_dict[fid].iloc[0]['event_date']
            return (datetime.strptime(as_of_date, "%Y-%m-%d") -
                    datetime.strptime(first, "%Y-%m-%d")).days / 365.25
        return 0
    row['diff_ufc_age'] = get_ufc_age(f1_id) - get_ufc_age(f2_id)

    row['age_td_interaction']         = row.get('age_diff', 0) * row.get('diff_td_avg_dec_avg', 0)
    row['age_td_adjperf_interaction'] = row.get('age_diff', 0) * row.get('diff_td_avg_adjperf', 0)
    row['age_str_interaction']        = row.get('age_diff', 0) * row.get('diff_str_acc_dec_avg', 0)
    row['age_winratio_interaction']   = row.get('age_diff', 0) * row.get('diff_win_ratio', 0)
    row['ufc_age_td_interaction']     = row.get('diff_ufc_age', 0) * row.get('diff_td_avg_dec_avg', 0)
    row['ufc_age_str_interaction']    = row.get('diff_ufc_age', 0) * row.get('diff_str_acc_dec_avg', 0)
    row['td_ctrl_interaction']        = row.get('diff_td_avg_dec_avg', 0) * row.get('diff_ctrl_time_per_min_dec_avg', 0)
    row['str_headdef_interaction']    = row.get('diff_str_acc_dec_avg', 0) * row.get('diff_head_allowed_dec_avg', 0)

    X_pred = pd.DataFrame([row])
    for col in feature_cols:
        if col not in X_pred.columns:
            X_pred[col] = 0.0
    X_pred = X_pred[feature_cols].fillna(0)

    prob   = model.predict_proba(X_pred)[0]
    pred   = model.predict(X_pred)[0]
    winner = f1_db_name if pred == 1 else f2_db_name
    conf   = max(prob) * 100

    print(f"\n  {'='*50}")
    print(f"  {f1_db_name:>25} | {prob[1]*100:5.1f}%")
    print(f"  {f2_db_name:>25} | {prob[0]*100:5.1f}%")
    print(f"  {'─'*50}")
    print(f"  Pick: {winner} ({conf:.1f}%)")
    print(f"  {'='*50}\n")

    return {'f1': f1_db_name, 'f2': f2_db_name,
            'f1_prob': prob[1], 'f2_prob': prob[0],
            'pick': winner, 'confidence': conf}

In [8]:
predict_fight("Jiri Prochazka", "Carlos Ulberg", conn)
predict_fight("Azamat Murzakanov ", "Paulo Costa", conn)
predict_fight("Curtis Blaydes", "Josh Hokit", conn)
predict_fight("Dominick Reyes", "Dominick Reyes", conn)
predict_fight("Cub Swanson", "Nate Landwehr", conn)


  F1: Jiri Prochazka (009341ed974bad72)
  F2: Carlos Ulberg (9014c02eff8b3d62)


NameError: name 'calculate_three_layer_features_v2' is not defined